# Experiment 3: Evidence Preservation and Correctable AI Memory
**Frozen protocol implementation**

Primary comparison:
* **A:** mutable atomic memory
* **EI:** immutable human evidence plus mutable derived interpretation

The notebook separately records active fidelity, evidence integrity, recovery, behavior, boundary behavior, and deliberate corruption/repair. Final scientific scoring remains manual against the frozen ground truth.


In [ ]:
!pip -q install transformers accelerate sentencepiece


In [ ]:
import os, json, copy, csv, shutil
from pathlib import Path
from datetime import datetime, timezone
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed

MODEL_NAME="Qwen/Qwen2.5-1.5B-Instruct"
SEED=42
MAX_UPDATE_TOKENS=700
MAX_RESPONSE_TOKENS=400
MAX_RECOVERY_TOKENS=500
MAX_MEMORY_TOKENS=4000
DO_SAMPLE=False
USE_GOOGLE_DRIVE=True

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR=Path("/content/drive/MyDrive/Sarah_Memory_Experiment_3")
else:
    OUTPUT_DIR=Path("/content/Sarah_Memory_Experiment_3")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

set_seed(SEED)
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME)
model=AutoModelForCausalLM.from_pretrained(MODEL_NAME,torch_dtype="auto",device_map="auto")
model.eval()

def tc(x): return len(tokenizer.encode(x,add_special_tokens=False))
def gen(system,user,limit,label):
    msgs=[{"role":"system","content":system},{"role":"user","content":user}]
    txt=tokenizer.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True)
    inp=tokenizer(txt,return_tensors="pt").to(model.device)
    with torch.no_grad():
        o=model.generate(**inp,max_new_tokens=limit,do_sample=DO_SAMPLE,pad_token_id=tokenizer.eos_token_id)
    new=o[0][inp["input_ids"].shape[1]:]
    if len(new)>=limit: raise RuntimeError(f"{label} reached token ceiling {limit}/{limit}; do not silently truncate.")
    return tokenizer.decode(new,skip_special_tokens=True).strip()
print("Ready:",MODEL_NAME,OUTPUT_DIR)


## Frozen protocol
No hypothesis, event, checkpoint, ground truth, probe, metric, or falsification criterion should be changed after results are inspected without recording it as a protocol amendment.


In [ ]:
EVENTS=[
(1,"M1","When I ask you to explain something technical, start with a simple explanation before adding technical detail."),
(2,"M2","When I'm choosing between tools, I usually want one recommendation."),
(3,"M3","If I postpone scheduling a meeting, don't assume I'm uninterested. I often postpone because my calendar is uncertain."),
(4,"M4","When I pause a creative project, ask what is blocking me before suggesting a solution."),
(5,"M5","When an experiment produces an unexpected result, give me multiple plausible explanations."),
(6,"U6","I prefer short email subject lines."),
(7,"M2","I've changed my mind. When I'm choosing between tools, give me the best two options."),
(8,"U8","I prefer research sessions in the morning."),
(9,"M1","If I explicitly ask for the technical version, skip the simple explanation."),
(10,"U10","I prefer concise presentation titles."),
(11,"M3","If I explicitly tell you why I postponed the meeting, use that reason instead of the calendar explanation."),
(12,"U12","I like written preparation before meetings."),
(13,"M2","Another update: I now want three options with the main difference between them."),
(14,"U14","I organize research notes by question."),
(15,"M4","If I say I'm intentionally taking a break, don't treat the pause as a problem that needs solving."),
(16,"U16","I prefer short conference follow ups."),
(17,"M5","If the result is explained by a confirmed implementation bug, don't generate additional explanations unless I ask."),
(18,"U18","I prefer tables for direct comparisons."),
(19,"U19","I prefer action items at the end of project notes."),
(20,"U20","I separate experiment conclusions from hypotheses.")]

TARGET=["M1","M2","M3","M4","M5"]; CPS=[5,10,15,20]
GT={
5:{"M1":"Technical explanations start simple before adding technical detail.","M2":"When choosing between tools, give one recommendation.","M3":"Meeting postponement should not imply disinterest; calendar uncertainty is the default stated explanation.","M4":"For a paused creative project, ask what is blocking Sarah before suggesting a solution.","M5":"For an unexpected experimental result, give multiple plausible explanations."},
10:{"M1":"Technical explanations start simple, except when Sarah explicitly asks for the technical version, in which case skip the simple explanation.","M2":"When choosing between tools, give the best two options.","M3":"Meeting postponement should not imply disinterest; calendar uncertainty is the default stated explanation.","M4":"For a paused creative project, ask what is blocking Sarah before suggesting a solution.","M5":"For an unexpected experimental result, give multiple plausible explanations."},
15:{"M1":"Technical explanations start simple unless Sarah explicitly asks for the technical version.","M2":"When choosing between tools, give three options with the main difference between them.","M3":"Meeting postponement should not imply disinterest. Calendar uncertainty is the default, but an explicitly supplied reason takes precedence.","M4":"Ask about blockers for an unexplained creative pause, but an intentional break should not be treated as a problem needing a solution.","M5":"For an unexpected experimental result, give multiple plausible explanations."},
20:{"M1":"Technical explanations start simple unless Sarah explicitly asks for the technical version.","M2":"When choosing between tools, give three options with the main difference between them.","M3":"Meeting postponement should not imply disinterest. Calendar uncertainty is the default, but an explicitly supplied reason takes precedence.","M4":"Ask about blockers for an unexplained creative pause, but an intentional break should not be treated as a problem needing a solution.","M5":"Give multiple plausible explanations for an unexpected experimental result unless a confirmed implementation bug explains it; then do not generate additional explanations unless Sarah asks."}}
assert [x[0] for x in EVENTS]==list(range(1,21))


In [ ]:
UPDATE_SYS="""Revise ONE memory record about Sarah. Preserve every still-valid part. Incorporate only what Sarah explicitly changes, qualifies, supersedes, or adds. Do not infer unstated traits or motives. Return only the complete revised rule."""
EI_SYS="""Derive the CURRENT rule from ordered immutable statements by Sarah. Later explicit corrections can supersede earlier instructions. Preserve unchanged earlier parts. Keep exceptions and boundaries. Do not blend superseded preferences. Return only the current rule."""

def run_A():
    mem={}; hist=[]; cps={}
    for turn,rid,s in EVENTS:
        before=copy.deepcopy(mem)
        if rid not in mem: mem[rid]=s; op="direct_insert"
        else: mem[rid]=gen(UPDATE_SYS,f"ID {rid}\nEXISTING:\n{mem[rid]}\nNEW HUMAN STATEMENT:\n{s}\nReturn current rule.",MAX_UPDATE_TOKENS,f"A {turn}")
        for k,v in before.items():
            if k!=rid: assert mem[k]==v
        if tc(json.dumps(mem,ensure_ascii=False))>MAX_MEMORY_TOKENS: raise RuntimeError("A memory budget exceeded")
        hist.append({"turn":turn,"id":rid,"statement":s,"operation":op if rid not in before else "local_revision","memory_after":copy.deepcopy(mem)})
        if turn in CPS: cps[turn]=copy.deepcopy(mem)
    return {"history":hist,"checkpoints":cps,"final":mem}

def run_EI():
    ledger=[]; interp={}; hist=[]; cps=[]
    cpdict={}; eid=0
    for turn,rid,s in EVENTS:
        eid+=1; ledger.append({"evidence_id":eid,"turn":turn,"record_id":rid,"statement":s})
        rel=[e for e in ledger if e["record_id"]==rid]
        if len(rel)==1: interp[rid]=s
        else:
            ev="\n".join(f"E{e['evidence_id']} turn {e['turn']}: {e['statement']}" for e in rel)
            interp[rid]=gen(EI_SYS,f"ID {rid}\nEVIDENCE:\n{ev}",MAX_UPDATE_TOKENS,f"EI {turn}")
        # immutable evidence invariant
        for e in ledger:
            _,er,es=EVENTS[e["turn"]-1]
            assert e["record_id"]==er and e["statement"]==es
        if tc(json.dumps({"ledger":ledger,"interpretation":interp},ensure_ascii=False))>MAX_MEMORY_TOKENS: raise RuntimeError("EI memory budget exceeded")
        hist.append({"turn":turn,"id":rid,"statement":s,"ledger_after":copy.deepcopy(ledger),"interpretation_after":copy.deepcopy(interp)})
        if turn in CPS: cpdict[turn]={"ledger":copy.deepcopy(ledger),"interpretation":copy.deepcopy(interp)}
    return {"history":hist,"checkpoints":cpdict,"final":{"ledger":ledger,"interpretation":interp}}

A=run_A(); EI=run_EI()
print("Primary memory runs complete")


In [ ]:
# Mechanical evidence integrity
EVIDENCE_INTEGRITY=[]
for cp in CPS:
    stored=[(e["turn"],e["record_id"],e["statement"]) for e in EI["checkpoints"][cp]["ledger"]]
    expected=[x for x in EVENTS if x[0]<=cp]
    EVIDENCE_INTEGRITY.append({"checkpoint":cp,"exact_match":stored==expected,"score":1 if stored==expected else 0})
EVIDENCE_INTEGRITY


In [ ]:
# Recovery agent sees only immutable evidence, never active interpretation.
REC_SYS="""Reconstruct Sarah's CURRENT rule using only the ordered immutable human evidence. Later explicit changes may supersede earlier statements. Preserve unchanged parts. Distinguish current from superseded instructions. Do not infer unstated information. Return only the reconstructed current rule."""
RECOVERY={}
for cp in CPS:
    RECOVERY[cp]={}
    led=EI["checkpoints"][cp]["ledger"]
    for mid in TARGET:
        rel=[e for e in led if e["record_id"]==mid]
        ev="\n".join(f"E{e['evidence_id']} turn {e['turn']}: {e['statement']}" for e in rel)
        RECOVERY[cp][mid]=gen(REC_SYS,f"ID {mid}\nEVIDENCE:\n{ev}",MAX_RECOVERY_TOKENS,f"recovery {cp} {mid}")
print("Recovery complete")


In [ ]:
BEHAVIOR_PROBES={
"M1":"I need an explanation of how vector indexes work, and I explicitly want the technical version. How would you explain it?",
"M2":"I'm choosing between three note-taking systems. How many options should you present, and how should you structure them?",
"M3":"I postponed our meeting because I am waiting for the finance team to send me the numbers. What should you infer about why I postponed it?",
"M4":"I've stopped working on my screenplay for two weeks because I've intentionally decided to take a break. What should we do about the blockage?",
"M5":"Our unexpected result turned out to come from a confirmed indexing bug. What alternative scientific explanations should we investigate?"}
B_SYS="""Respond directly to Sarah using supplied active memory where relevant. Do not invent preferences or motives. Respond in no more than 100 words."""
def memtxt(cond,cp):
    m=A["checkpoints"][cp] if cond=="A" else EI["checkpoints"][cp]["interpretation"]
    return "\n".join(f"{k}: {v}" for k,v in sorted(m.items()))
BEHAVIOR={}
for cond in ["A","EI"]:
    BEHAVIOR[cond]={}
    for cp in CPS:
        BEHAVIOR[cond][cp]={}
        for mid,p in BEHAVIOR_PROBES.items():
            BEHAVIOR[cond][cp][mid]=gen(B_SYS,f"ACTIVE MEMORY:\n{memtxt(cond,cp)}\n\nSARAH:\n{p}",MAX_RESPONSE_TOKENS,f"behavior {cond} {cp} {mid}")
print("Behavior complete")


In [ ]:
# Critical historical contradiction test: one -> two -> three.
CONTRADICTION={}
p="I'm comparing project-management tools. Follow my current preference for how many options to give me and how to distinguish them."
for cond in ["A","EI"]:
    CONTRADICTION[cond]=gen(B_SYS,f"ACTIVE MEMORY:\n{memtxt(cond,20)}\n\nSARAH:\n{p}",MAX_RESPONSE_TOKENS,f"contradiction {cond}")
CONTRADICTION


In [ ]:
# Deliberate corruption and repair. Primary run remains untouched.
CORRUPTED="When Sarah is choosing between tools, give her two options."
REPAIR_SYS="""Repair the corrupted current memory using only the architecture-specific information supplied. Do not use outside assumptions. Return only the repaired current rule, or state that reconstruction is not possible from available memory."""

A_REPAIR=gen(REPAIR_SYS,f"CORRUPTED MUTABLE M2:\n{CORRUPTED}\nNo earlier M2 evidence is available in this architecture for this repair test.",MAX_RECOVERY_TOKENS,"A repair")
rel=[e for e in EI["checkpoints"][20]["ledger"] if e["record_id"]=="M2"]
ev="\n".join(f"E{e['evidence_id']} turn {e['turn']}: {e['statement']}" for e in rel)
EI_REPAIR=gen(REPAIR_SYS,f"CORRUPTED CURRENT M2:\n{CORRUPTED}\n\nIMMUTABLE M2 EVIDENCE:\n{ev}",MAX_RECOVERY_TOKENS,"EI repair")
REPAIR={"corrupted":CORRUPTED,"A":A_REPAIR,"EI":EI_REPAIR,"ground_truth":GT[20]["M2"]}
REPAIR


In [ ]:
BOUNDARY_PROBES=[
"I'm choosing between three routes for my walk. How many should you tell me about?",
"I postponed watering my plants until tomorrow. Does that mean my calendar is uncertain?",
"What is the capital of Finland? Please answer directly."]
BOUNDARY={}
for cond in ["A","EI"]:
    BOUNDARY[cond]=[]
    for i,p in enumerate(BOUNDARY_PROBES,1):
        BOUNDARY[cond].append({"id":f"B{i}","prompt":p,"response":gen(B_SYS,f"ACTIVE MEMORY:\n{memtxt(cond,20)}\n\nSARAH:\n{p}",MAX_RESPONSE_TOKENS,f"boundary {cond} {i}")})
print("Boundary complete")


## Manual scoring
Use **1 = faithful**, **0.5 = partially faithful**, **0 = materially incorrect, absent, or contradicted**.

Active Fidelity is scored from the raw active representation, not behavior. Recovery Fidelity is scored from the independent Recovery Agent output. Behavior remains a separate dependent variable.

Possible failure labels: `LOSS`, `DISTORTION`, `BOUNDARY`, `OBSOLETE`, `UNSUPPORTED_INFERENCE`, `PROVENANCE`, `RETRIEVAL`, `APPLICATION`.


In [ ]:
rows=[]
for cond in ["A","EI"]:
    for cp in CPS:
        for mid in TARGET:
            obs=A["checkpoints"][cp].get(mid,"") if cond=="A" else EI["checkpoints"][cp]["interpretation"].get(mid,"")
            rows.append({"measure":"active_fidelity","condition":cond,"checkpoint":cp,"memory_id":mid,"ground_truth":GT[cp][mid],"observed":obs,"score_0_05_1":"","error_label":"","reason":""})
for cp in CPS:
    for mid in TARGET:
        rows.append({"measure":"recovery_fidelity","condition":"EI","checkpoint":cp,"memory_id":mid,"ground_truth":GT[cp][mid],"observed":RECOVERY[cp][mid],"score_0_05_1":"","error_label":"","reason":""})
scorefile=OUTPUT_DIR/"manual_scoring_template.csv"
with open(scorefile,"w",newline="",encoding="utf-8") as f:
    w=csv.DictWriter(f,fieldnames=rows[0].keys()); w.writeheader(); w.writerows(rows)
print(scorefile)


In [ ]:
RESULTS={
"metadata":{"experiment":"Experiment 3: Evidence Preservation and Correctable AI Memory","model":MODEL_NAME,"seed":SEED,"protocol_status":"frozen_before_results","timestamp_utc":datetime.now(timezone.utc).isoformat()},
"events":EVENTS,"ground_truth":GT,"condition_A":A,"condition_EI":EI,
"evidence_integrity":EVIDENCE_INTEGRITY,"recovery":RECOVERY,
"behavior_probes":BEHAVIOR_PROBES,"behavior":BEHAVIOR,
"contradiction":CONTRADICTION,"corruption_repair":REPAIR,"boundary":BOUNDARY}
with open(OUTPUT_DIR/"experiment_3_results.json","w",encoding="utf-8") as f: json.dump(RESULTS,f,ensure_ascii=False,indent=2)
with open(OUTPUT_DIR/"frozen_protocol_snapshot.json","w",encoding="utf-8") as f:
    json.dump({"events":EVENTS,"ground_truth":GT,"checkpoints":CPS,"behavior_probes":BEHAVIOR_PROBES,"boundary_probes":BOUNDARY_PROBES,"corrupted_M2":CORRUPTED},f,ensure_ascii=False,indent=2)
zip_path=shutil.make_archive(str(OUTPUT_DIR/"Sarah_Memory_Experiment_3_Results"),"zip",root_dir=OUTPUT_DIR)
print("Saved results ZIP:",zip_path)


## Before interpretation
Keep the ZIP unchanged as the raw run record. Do not change ground truth after inspecting results. Any technical correction must be documented before rerunning affected components.

The central test is whether separating evidence from interpretation makes memory **more correctable without creating harmful historical interference**.
